# ⚡ Cloud Ultra Downloader | منصة التحميل السحابي فائق السرعة

[![GitHub](https://img.shields.io/badge/GitHub-ElkampaPro%2Fcloud--ultra--downloader-indigo?logo=github)](https://github.com/ElkampaPro/cloud-ultra-downloader)
![Aria2 16 Threads](https://img.shields.io/badge/Engine-Aria2c%2016%20Streams-cyan)
![Google Drive 5TB](https://img.shields.io/badge/Storage-Google%20Drive%205TB-emerald)

> **التحميل السحابي الصاروخي وحفظ الملفات مباشرة في Google Drive**  
> يدعم الروابط المباشرة، فهارس مواقع الأنمي (MugiSubs / Kiyoshii / GoIndex)، OneDrive، وروابط التورنت والمغناطيس (Magnet).

---

### الخطوة 1: ربط Google Drive (لحفظ الملفات مباشرة)
قم بتشغيل الخلية أدناه وامنح الإذن لربط Google Drive.

In [ ]:
from google.colab import drive
print("⚡ جارٍ ربط Google Drive...")
drive.mount('/content/drive')

import os
DOWNLOAD_DIR = '/content/drive/MyDrive/Downloads'
os.makedirs(DOWNLOAD_DIR, exist_ok=True)
print(f"✓ تم إعداد مجلد التنزيلات بنجاح: {DOWNLOAD_DIR}")

### الخطوة 2: تثبيت الحزم ومحرك التحميل السحابي (Aria2c)
تثبيت `aria2` ومكتبات بايثون وسحب أحدث كود للمشروع.

In [ ]:
# 1. Install aria2c
!apt-get update -qq && apt-get install -y -qq aria2

# 2. Clone or Update Repository
import os
REPO_URL = "https://github.com/ElkampaPro/cloud-ultra-downloader.git"
PROJECT_DIR = "/content/cloud-ultra-downloader"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    %cd {PROJECT_DIR}
    !git pull

%cd {PROJECT_DIR}
!pip install -q -r requirements.txt
print("✓ اكتمل تجهيز البيئة بنجاح!")

### الخطوة 3: تشغيل محرك Aria2c + سيرفر الويب واستخراج رابط الواجهة
قم بتشغيل هذه الخلية وستحصل على رابط HTTPS مباشر تفتحه من هاتفك أو كمبيوترك!

In [ ]:
import subprocess, time, re, os, urllib.request
from IPython.display import display, HTML

%cd /content/cloud-ultra-downloader

# 1. Start Aria2c Daemon (16 parallel connections per file)
DOWNLOAD_DIR = "/content/drive/MyDrive/Downloads"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)
!pkill -f aria2c || true
!pkill -f uvicorn || true
!pkill -f cloudflared || true

aria2_cmd = [
    "aria2c",
    "--enable-rpc",
    "--rpc-listen-all=true",
    "--rpc-listen-port=6800",
    "--max-connection-per-server=16",
    "--split=16",
    "--min-split-size=1M",
    "--max-concurrent-downloads=5",
    "--continue=true",
    f"--dir={DOWNLOAD_DIR}",
    "--rpc-allow-origin-all=true",
    "--daemon=true"
]
subprocess.Popen(aria2_cmd)
time.sleep(1)
print("✓ تم تشغيل محرك Aria2c السحابي (16 مسار اتصال)")

# 2. Start FastAPI Web Server in Background
web_proc = subprocess.Popen(
    ["python3", "-m", "uvicorn", "app.main:app", "--host", "127.0.0.1", "--port", "8000"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(2)
print("✓ تم تشغيل سيرفر الويب على المنفذ 8000")

# 3. Download & Run Cloudflare Tunnel for secure instant HTTPS link
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("⚡ جارٍ تنزيل نفق Cloudflare الآمن...")
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

cf_proc = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

tunnel_url = None
start_time = time.time()
while time.time() - start_time < 25:
    line = cf_proc.stdout.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(0.5)

if tunnel_url:
    html_content = f"""
    <div style="padding: 20px; background: linear-gradient(135deg, #090d16 0%, #1e1b4b 100%); border-radius: 16px; border: 1px solid #4338ca; color: white; font-family: 'Cairo', sans-serif; text-align: center; margin-top: 15px;">
        <h2 style="margin: 0 0 10px 0; color: #38bdf8;">⚡ تم تشغيل لوحة التحكم السحابية بنجاح!</h2>
        <p style="margin: 0 0 15px 0; font-size: 14px; color: #cbd5e1;">افتح الرابط أدناه من هاتفك أو حاسوبك وابدأ التحميل مباشرة إلى درايف:</p>
        <a href="{tunnel_url}" target="_blank" style="display: inline-block; padding: 12px 28px; background: linear-gradient(90deg, #6366f1, #4f46e5); color: white; text-decoration: none; border-radius: 10px; font-weight: bold; font-size: 16px; box-shadow: 0 4px 15px rgba(99, 102, 241, 0.4);">🚀 فتح لوحة التحكم (Open Web Dashboard)</a>
        <p style="margin: 15px 0 0 0; font-size: 12px; color: #94a3b8; font-family: monospace;">{tunnel_url}</p>
    </div>
    """
    display(HTML(html_content))
else:
    print("لم نتمكن من التقاط رابط Cloudflare، جرب الرابط المحلي أو نفق Pinggy:")
    !ssh -p 443 -R0:localhost:8000 -o StrictHostKeyChecking=no a.pinggy.io
